In [ ]:
#将所有的预测结果按照流域分开，每个流域单独一个csv文件
# 输入一个大 CSV 文件，按照第一列 no 的值（比如每个流域编号）分开，输出为每个流域单独一个csv文件
import pandas as pd
import os

# 输入和输出路径
input_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\p12\\p12ALL.csv'
output_folder = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\esm-ssp585-ALL-basins'

# 确保输出文件夹存在
os.makedirs(output_folder, exist_ok=True)

# 读取CSV文件
df = pd.read_csv(input_file)

# 要保留的列
columns_to_save = ['year', 'PPT', 'PET', 'R', 'n', 'AET', 'LAI', 'treeFrac']

# 遍历每个不同的 'no'
for no_value, group in df.groupby(df.columns[0]):  # 第一列一般是 'no'
    # 只保留需要的列
    group_selected = group[columns_to_save]
    
    # 生成输出文件路径
    output_path = os.path.join(output_folder, f"{no_value}.csv")
    
    # 保存到对应的文件
    group_selected.to_csv(output_path, index=False)

print("所有流域文件已保存到：", output_folder)


In [ ]:
#用于绘制空间图
# 对所有流域最后一个窗口归因（变化期2071--2100），并计算年代变化和变化率（2031--2040、2061--2070、2091--2100）
import pandas as pd
import numpy as np
import math
import os

# Budyko 模型函数
def budyko(PPT, PET, n):
    phi = PET / PPT
    ET = PPT * (1 + phi - (1 + phi**n)**(1/n))
    R = PPT - ET
    return R

# 弹性系数计算
def compute_elasticities(PPT, PET, n):
    phi = PET / PPT
    eps_ppt = ((1 + phi**n)**(1/n+1) - phi**(n+1)) / ((1 + phi**n) * ((1 + phi**n)**(1/n) - phi))
    eps_pet = 1 / ((1 + phi**n) * (1 - (1 + phi**-n)**(1/n)))
    eps_n = (math.log(1 + phi**n) + phi**n * math.log(1 + phi**-n)) / (n * (1 + phi**n) * (1 - (1 + phi**-n)**(1/n)))
    return eps_ppt, eps_pet, eps_n

# 路径设置
input_folder = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\esm-ssp585-ALL-basins'
output_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\guiyin\\esm-ssp585-ALL.csv'

all_results = []

# 文件遍历
for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_folder, filename)
        df = pd.read_csv(file_path)

        # 全时段平均
        all_period = df[(df['year'] >= 1985) & (df['year'] <= 2100)]
        PPT00 = all_period['PPT'].mean()
        PET00 = all_period['PET'].mean()
        n00 = all_period['n'].mean()
        R00 = all_period['R'].mean()
        eps_ppt00, eps_pet00, eps_n00 = compute_elasticities(PPT00, PET00, n00)

        # 基准期
        base_period = df[(df['year'] >= 1985) & (df['year'] <= 2014)]
        PPT0 = base_period['PPT'].mean()
        PET0 = base_period['PET'].mean()
        n0 = base_period['n'].mean()
        R0 = base_period['R'].mean()
        tree_base = base_period['treeFrac'].mean()
        hisR = R0

        # 变化期
        change_period = df[(df['year'] >= 2071) & (df['year'] <= 2100)]
        PPT1 = change_period['PPT'].mean()
        PET1 = change_period['PET'].mean()
        n1 = change_period['n'].mean()
        R1 = change_period['R'].mean()
        dis_n = n1 - n0
        eps_ppt1, eps_pet1, eps_n1 = compute_elasticities(PPT1, PET1, n1)

        # ΔR 分解
        delta_PPT = PPT1 - PPT0
        delta_PET = PET1 - PET0
        delta_n = n1 - n0
        delta_RPPT = delta_PPT / PPT00 * eps_ppt00 * R00
        delta_RPET = delta_PET / PET00 * eps_pet00 * R00
        delta_Rn = delta_n / n00 * eps_n00 * R00
        delta_RCC = delta_RPPT + delta_RPET
        delta_R_total = delta_RPPT + delta_RPET + delta_Rn

        if delta_R_total != 0:
            delta_PPT_percent = delta_RPPT / delta_R_total * 100
            delta_PET_percent = delta_RPET / delta_R_total * 100
            delta_CC_percent = delta_PPT_percent + delta_PET_percent
            delta_n_percent = delta_Rn / delta_R_total * 100
        else:
            delta_PPT_percent = delta_PET_percent = delta_n_percent = np.nan
            delta_CC_percent = np.nan

        # 时间窗口
        windows = {
            '4': (2031, 2040),
            '7': (2061, 2070),
            '10': (2091, 2100),
        }

        base_vars = base_period[['LAI', 'PPT', 'AET', 'R']].mean()
        variation_values = []
        rate_values = []
        tree_values = []
        tree_deltas = []

        for label, (start, end) in windows.items():
            future_period = df[(df['year'] >= start) & (df['year'] <= end)]
            if future_period.empty:
                variation_values.extend([np.nan] * 4)
                rate_values.extend([np.nan] * 4)
                tree_values.append(np.nan)
                tree_deltas.append(np.nan)
            else:
                future_mean = future_period[['LAI', 'PPT', 'AET', 'R']].mean()
                delta = future_mean - base_vars
                rate = delta / base_vars * 100
                variation_values.extend(delta.values.tolist())
                rate_values.extend(rate.values.tolist())

                tree = future_period['treeFrac'].mean()
                tree_values.append(tree)
                tree_deltas.append(tree - tree_base)

        # 汇总
        all_results.append([
            os.path.splitext(filename)[0],
            eps_ppt1, eps_pet1, eps_n1,
            delta_RPPT, delta_RPET, delta_RCC, delta_Rn, delta_R_total,
            delta_PPT_percent, delta_PET_percent, delta_CC_percent, delta_n_percent,
            hisR, dis_n
        ] + variation_values + rate_values + tree_values + tree_deltas)

# 构建列名，eps_xx：弹性；dRxx：各要素引起的径流变化量（▲Rxx）；pxx：各要素的贡献
# hisR：历史时期（1985--2014）平均径流量；dis_n：变化期（2071--2100）和基准期（1985--2014）n的差值
columns = [
    'Filename',
    'eps_PPT', 'eps_PET', 'eps_n',
    'dRPPT', 'dRPET', 'dRCC', 'dRn', 'dR',
    'pPPT', 'pPET', 'pCC', 'pn',
    'hisR', 'dis_n'
]

# v_xx代表变化量；r_xx代表变化率；
#tree_代表森林覆盖率；v_tree_xx代表森林覆盖率变化量；
#'4'代表(2031, 2040)；'7'代表(2061, 2070)；'10'代表(2091, 2100)
vars_ = ['LAI', 'PPT', 'AET', 'R']
for label in ['4', '7', '10']:
    columns += [f'v_{label}_{v}' for v in vars_]
for label in ['4', '7', '10']:
    columns += [f'r_{label}_{v}' for v in vars_]
columns += [f'tree_{label}' for label in ['4', '7', '10']]
columns += [f'v_tree_{label}' for label in ['4', '7', '10']]

# 保存
output_df = pd.DataFrame(all_results, columns=columns)
output_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"全部计算完成！结果保存到 {output_file}")


In [ ]:
#写入shp文件
import geopandas as gpd
import pandas as pd
import os

# 输入路径
csv_file = r'D:\DESKTOP\desk\bianliang\\esm-ssp585\\attribution\\guiyin\\esm-ssp585-ALL.csv'
shp_folder = r'D:\DESKTOP\desk\\private\\map\BasinATLAS_Data_v10.gdb&shp\BasinATLAS5'
output_shp = r'D:\DESKTOP\desk\\private\\map\BasinATLAS_Data_v10.gdb&shp\\attribution\\esm-ssp585'

os.makedirs(output_shp, exist_ok=True)

# 读取CSV
df_csv = pd.read_csv(csv_file)

# 存储所有更新后的GeoDataFrame
gdf_list = []

for idx, row in df_csv.iterrows():
    filename = row['Filename']
    shp_path = os.path.join(shp_folder, f"{filename}.shp")
    
    if not os.path.exists(shp_path):
        print(f"警告：{shp_path} 不存在，跳过。")
        continue
    
    # 读取对应的shp
    gdf = gpd.read_file(shp_path)
    
    # 将CSV中的数据写入shp的属性表
    for col in df_csv.columns[1:]:  # 跳过Filename列
        gdf[col] = row[col]
    
    # 保存到列表
    gdf_list.append(gdf)

# 合并所有gdf
if gdf_list:
    merged_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True), crs=gdf_list[0].crs)
    merged_gdf.to_file(output_shp, encoding='utf-8-sig')
    print(f"全部完成, 合并后的shp保存到: {output_shp}")
else:
    print("没有有效的shp文件被处理。")
